In [1]:
from pathlib import Path
import subprocess
import sys
import torch

In [ ]:
print(torch.__version__)

2.11.0+cpu


We’ll proceed in this order:
1. Download the repository and check dependencies.
2. Install dependencies and verify imports.
3. Inspect the transcoder: encoder, sparse activations, decoder, and loss.
4. Run a small training test and understand its inputs, targets, and outputs.
5. Add the frozen-encoder and frozen-decoder variants.

In [2]:
repo_path = Path("/content/transcoder_circuits")

if not repo_path.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/jacobdunefsky/transcoder_circuits.git",
            str(repo_path),
        ],
        check=True,
    )

commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=repo_path,
    text=True,
).strip()

print("Repository:", repo_path)
print("Commit:", commit)
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

print("\nRepository requirements:")
print((repo_path / "requirements.txt").read_text())

Repository: /content/transcoder_circuits
Commit: 8721ff06b64889e8cc50bb1c5e869c7f72b8303d
Python: 3.13.16
PyTorch: 2.11.0+cpu
GPU available: False

Repository requirements:
matplotlib
numpy
plotly
torch==2.2.0
tqdm
transformer_lens==1.11.0
datasets==2.12.0
einops==0.7.0
setuptools==67.7.2
wandb==0.16.0
huggingface-hub==0.17.3


In [ ]:
training_path = repo_path / "sae_training"

print("Training files:")
for path in sorted(training_path.glob("*.py")):
    print(" -", path.name)

print("\nImports used by the training package:")
for path in sorted(training_path.glob("*.py")):
    imports = [
        line.strip()
        for line in path.read_text().splitlines()
        if line.startswith(("import ", "from "))
    ]
    if imports:
        print(f"\n{path.name}")
        print("\n".join(imports))

Training files:
 - __init__.py
 - activations_store.py
 - config.py
 - optim.py
 - sparse_autoencoder.py
 - train_sae_on_language_model.py
 - utils.py

Imports used by the training package:

activations_store.py
import os
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformer_lens import HookedTransformer
import gc

config.py
from abc import ABC
from dataclasses import dataclass
from typing import Optional
import torch
import wandb

optim.py
import math
from typing import Optional
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler

sparse_autoencoder.py
import gzip
import os
import pickle
from functools import partial
import einops
import torch
import torch.nn.functional as F
from jaxtyping import Float
from torch import Tensor, nn
from torch.distributions.categorical import Categorical
from tqdm import tqdm
from transformer_lens.hook_points import HookedRootModule, HookPoint
from .geom_medi

In [ ]:
print((repo_path / "setup.sh").read_text())

echo "Installing Python packages."

pip install -r requirements.txt

echo "Package installation complete."

echo "Checking whether transcoders present."
DIR_NAME='./gpt-2-small-transcoders'
if [ ! -d "$DIR_NAME" ]; then
    mkdir "$DIR_NAME"
fi
if [ -z $(ls -A "$DIR_NAME") ]; then
    echo "Transcoders not found. Downloading transcoders."

    export HF_HUB_DISABLE_PROGRESS_BARS=1
    python - <<HERE
from huggingface_hub import snapshot_download
snapshot_download(repo_id="pchlenski/gpt2-transcoders", allow_patterns=["*.pt"],
    local_dir="$DIR_NAME", local_dir_use_symlinks=False
)
HERE
    export HF_HUB_DISABLE_PROGRESS_BARS=0
    echo "Transcoders downloaded."
fi




In [3]:
repo_path = Path("/content/transcoder_circuits")
# Preserve the currently installed PyTorch version.
constraints_path = repo_path / "pilot_constraints.txt"
constraints_path.write_text(f"torch=={torch.__version__}\n")

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--constraint", str(constraints_path),
        "transformer-lens==3.2.1",
        "einops",
        "jaxtyping",
        "wandb",
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--constraint', '/content/transcoder_circuits/pilot_constraints.txt', 'transformer-lens==3.2.1', 'einops', 'jaxtyping', 'wandb'], returncode=0)

In [4]:
#Check whether the repository’s classes import successfully
from importlib.metadata import version

# Let Python find the repository's sae_training package.
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

from transformer_lens.hook_points import HookedRootModule, HookPoint
from sae_training.config import LanguageModelSAERunnerConfig
from sae_training.sparse_autoencoder import SparseAutoencoder

print("TransformerLens:", version("transformer-lens"))
print("PyTorch:", version("torch"))
print("Repository imports passed.")

TransformerLens: 3.2.1
PyTorch: 2.11.0+cpu
Repository imports passed.


In [ ]:
import inspect

print("CONFIGURATION")
print(inspect.getsource(LanguageModelSAERunnerConfig))

print("\nTRANSCODER INITIALIZATION")
print(inspect.getsource(SparseAutoencoder.__init__))

print("\nFORWARD PASS")
print(inspect.getsource(SparseAutoencoder.forward))


CONFIGURATION
@dataclass
class LanguageModelSAERunnerConfig(RunnerConfig):
    """
    Configuration for training a sparse autoencoder on a language model.
    """

    # SAE Parameters
    b_dec_init_method: str = "geometric_median"
    expansion_factor: int = 4
    from_pretrained_path: Optional[str] = None

    # Training Parameters
    l1_coefficient: float = 1e-3
    lr: float = 3e-4
    lr_scheduler_name: str = "constant"  # constant, constantwithwarmup, linearwarmupdecay, cosineannealing, cosineannealingwarmup
    lr_warm_up_steps: int = 500
    train_batch_size: int = 4096

    # transcoder stuff
    is_transcoder: bool = False
    out_hook_point: Optional[str] = None
    out_hook_point_layer: Optional[int] = None
    d_out: Optional[int] = None

    # sparse-connection sparse transcoder stuff
    is_sparse_connection: bool = False
    sparse_connection_sae_path: Optional[str] = None
    sparse_connection_l1_coeff: Optional[float] = None
    sparse_connection_use_W_enc: bool = 

In [5]:
from sae_training.config import RunnerConfig


In [ ]:
import inspect
from sae_training.config import RunnerConfig

print(inspect.getsource(RunnerConfig))

@dataclass
class RunnerConfig(ABC):
    """
    The config that's shared across all runners.
    """

    # Data Generating Function (Model + Training Distibuion)
    model_name: str = "gelu-2l"
    hook_point: str = "blocks.0.hook_mlp_out"
    hook_point_layer: int = 0
    hook_point_head_index: Optional[int] = None
    dataset_path: str = "NeelNanda/c4-tokenized-2b"
    is_dataset_tokenized: bool = True
    context_size: int = 128
    use_cached_activations: bool = False
    cached_activations_path: Optional[
        str
    ] = None  # Defaults to "activations/{dataset}/{model}/{full_hook_name}_{hook_point_head_index}"

    # SAE Parameters
    d_in: int = 512

    # Activation Store Parameters
    n_batches_in_buffer: int = 20
    total_training_tokens: int = 2_000_000
    store_batch_size: int = 1024

    # Misc
    device: str = "cpu"
    seed: int = 42
    dtype: torch.dtype = torch.float32

    # transcoder stuff
    is_transcoder: bool = False
    out_hook_point: Optional[str]

In [ ]:
cfg = LanguageModelSAERunnerConfig(
    # Architecture
    is_transcoder=True,
    d_in=2,
    d_out=1,
    expansion_factor=4,        # 2 × 4 = 8 features

    # Execution
    device="cpu",
    dtype=torch.float32,
    seed=42,

    # Training settings
    l1_coefficient=1e-3,
    lr=3e-4,
    train_batch_size=32,

    # Keep the initial toy setup simple.
    b_dec_init_method="zeros",
    feature_sampling_method=None,
    use_ghost_grads=False,
    is_sparse_connection=False,
    log_to_wandb=False,

    # Small bookkeeping values for this toy configuration.
    context_size=1,
    store_batch_size=32,
    n_batches_in_buffer=1,
    total_training_tokens=3200,
)

# Explicitly seed PyTorch before initializing weights.
torch.manual_seed(cfg.seed)

transcoder = SparseAutoencoder(cfg)

print("\nParameter shapes:")
for name, parameter in transcoder.named_parameters():
    print(f"{name}: {tuple(parameter.shape)}")

assert transcoder.W_enc.shape == (2, 8)
assert transcoder.W_dec.shape == (8, 1)

Run name: 8-L1-0.001-LR-0.0003-Tokens-3.200e+03
n_tokens_per_buffer (millions): 3.2e-05
Lower bound: n_contexts_per_buffer (millions): 3.2e-05
Total training steps: 100
Total wandb updates: 10
n_tokens_per_feature_sampling_window (millions): 0.064
n_tokens_per_dead_feature_window (millions): 0.032
We will reset the sparsity calculation 0 times.
Number of tokens when resampling: 1024
Number tokens in sparsity calculation window: 6.40e+04

Parameter shapes:
W_enc: (2, 8)
b_enc: (8,)
W_dec: (8, 1)
b_dec: (2,)
b_dec_out: (1,)


**This is the core transcoder code.** It learns to map an input `x` to a target output `y`, using sparse intermediate features. Let’s understand that computation before creating it.

**1. The encoder calculates feature activations**

These lines implement:

\[
h=\operatorname{ReLU}\big((x-b_{\text{dec}})W_{\text{enc}}+b_{\text{enc}}\big)
\]

```python
sae_in = x - self.b_dec
hidden_pre = sae_in @ self.W_enc + self.b_enc
feature_acts = torch.relu(hidden_pre)
```

I’ve written the `einsum` operations as matrix multiplication to make them easier to read.

- `b_dec` shifts the input before encoding. Despite its name, in transcoder mode it serves as an **input-centering parameter**.
- `W_enc` maps the input into feature scores.
- `b_enc` shifts those scores.
- ReLU sets negative scores to zero, producing `feature_acts`.

For example, feature scores `[-2, 3, -1, 0.5]` become activations `[0, 3, 0, 0.5]`: **two active features**.

**2. The decoder produces the prediction**

With `is_transcoder=True`:

\[
\hat y=hW_{\text{dec}}+b_{\text{dec,out}}
\]

```python
sae_out = feature_acts @ self.W_dec + self.b_dec_out
```

Each feature contributes its activation strength multiplied by its decoder direction. Their contributions sum to form the predicted output.

For a toy transcoder with two input coordinates, eight features, and one output coordinate:

| Tensor | Shape |
|---|---|
| Input `x` | `[batch, 2]` |
| `W_enc` | `[2, 8]` |
| `feature_acts` | `[batch, 8]` |
| `W_dec` | `[8, 1]` |
| Prediction `sae_out` | `[batch, 1]` |

The configuration computes `d_sae = d_in * expansion_factor`, so `expansion_factor=4` gives eight features here.

**3. We must explicitly supply the target**

The intended call will be:

```python
outputs = transcoder(x, mse_target=y)
```

Here, `y` is the original computation’s output, such as `original(x)` in our toy tests or the GPT-2 MLP output later.

**Setting `is_transcoder=True` does not automatically select the target.** If `mse_target` is omitted, this implementation compares the prediction against `x`.

**4. The loss combines reconstruction and sparsity**

With ghost gradients disabled:

```python
loss = mse_loss + l1_loss
```

But its variable named `mse_loss` is **not ordinary MSE**. For a batch of vectors, it computes:

\[
L_{\text{reconstruction}}
=\frac{1}{BD_{\text{out}}}
\sum_{i,j}\frac{(\hat y_{ij}-y_{ij})^2}{\lVert y_i\rVert_2}
\]

It divides each example’s squared errors by that target’s norm, then averages.

The sparsity penalty is:

\[
L_{\text{sparsity}}
=\lambda\,\frac{1}{B}\sum_i\sum_k |h_{ik}|
\]

This penalizes activation magnitude. It encourages sparse features but **doesn’t fix the number of active features**.

**There are three details we need to handle before training:**

- **Zero targets:** the reconstruction denominator has no epsilon. An all-zero target can produce `NaN` or infinity—particularly relevant to our ReLU and OR examples.
- **Resampling default:** `"Anthropic"` is capitalized, but validation accepts lowercase `"anthropic"`. We’ll explicitly set `feature_sampling_method=None` for our initial test.
- **Ghost-gradient default:** `use_ghost_grads: bool = False,` has a trailing comma, making its default a nonempty tuple rather than a Boolean. We’ll explicitly pass `use_ghost_grads=False`.

One small comment is also misleading: the decoder normalization uses `dim=1`, so it normalizes each **row** of `W_dec`—each feature’s output direction.

**Next cell: inspect the inherited configuration defaults.** These contain settings such as input size, device, and dtype that weren’t included in your output:

```python
from sae_training.config import RunnerConfig

print(inspect.getsource(RunnerConfig))
```

Then we can construct a tiny CPU transcoder with explicit settings and handle the loss denominator before training.

> **Concept note:** A transcoder encodes an input into sparse features and decodes them into another computation’s output. Its training target must be supplied explicitly; reconstruction quality and sparsity are separate quantities.

In [ ]:
# Each row contains [u, m].
x = torch.tensor([
    [1.0, 1.0],
    [2.0, 2.0],
    [3.0, 3.0],
])

# Original function: f(u, m) = 2m.
y = 2 * x[:, 1:2]

transcoder.eval()

with torch.no_grad():
    (
        prediction,
        feature_acts,
        loss,
        reconstruction_loss,
        sparsity_loss,
        ghost_loss,
    ) = transcoder(x, mse_target=y)

print("Targets:\n", y)
print("Predictions:\n", prediction)
print("Feature activations:\n", feature_acts)

print("\nReconstruction loss:", reconstruction_loss.item())
print("Sparsity loss:", sparsity_loss.item())
print("Total loss:", loss.item())

assert prediction.shape == y.shape == (3, 1)
assert feature_acts.shape == (3, 8)
assert torch.isfinite(loss).item()

Targets:
 tensor([[2.],
        [4.],
        [6.]])
Predictions:
 tensor([[1.5610],
        [3.1220],
        [4.6830]])
Feature activations:
 tensor([[1.4255, 0.0835, 0.5499, 0.9576, 0.4501, 0.2920, 0.0000, 0.3863],
        [2.8511, 0.1669, 1.0997, 1.9153, 0.9002, 0.5841, 0.0000, 0.7727],
        [4.2766, 0.2504, 1.6496, 2.8729, 1.3502, 0.8761, 0.0000, 1.1590]])

Reconstruction loss: 0.19273202121257782
Sparsity loss: 0.008289895951747894
Total loss: 0.20102190971374512


In [ ]:
#Make the toy training loss work when the target is zero.
def toy_forward(model, x, target):
    x = x.to(device=model.W_enc.device, dtype=model.W_enc.dtype)
    target = target.to(device=x.device, dtype=x.dtype)

    # Same encoder and decoder computation as the repository.
    centered = model.hook_sae_in(x - model.b_dec)
    scores = model.hook_hidden_pre(centered @ model.W_enc + model.b_enc)
    features = model.hook_hidden_post(torch.relu(scores))
    prediction = model.hook_sae_out(
        features @ model.W_dec + model.b_dec_out
    )

    assert prediction.shape == target.shape

    reconstruction_loss = (prediction - target).square().mean()
    sparsity_loss = (
        model.l1_coefficient * features.abs().sum(dim=-1).mean()
    )
    loss = reconstruction_loss + sparsity_loss

    return prediction, features, loss, reconstruction_loss, sparsity_loss


with torch.no_grad():
    # Verify that the helper preserves our previous predictions/features.
    pred_check, feats_check, *_ = toy_forward(transcoder, x, y)
    torch.testing.assert_close(pred_check, prediction)
    torch.testing.assert_close(feats_check, feature_acts)

    # Test an example whose target is zero: m = 0, so 2m = 0.
    x_zero = torch.tensor([[1.0, 0.0]])
    y_zero = torch.tensor([[0.0]])

    pred_zero, _, loss_zero, _, _ = toy_forward(
        transcoder, x_zero, y_zero
    )

    assert torch.isfinite(loss_zero).item()

print("Predictions and features match the repository.")
print("Zero-target prediction:", pred_zero.item())
print("Zero-target loss:", loss_zero.item())
print("Zero-target check passed.")

NameError: name 'transcoder' is not defined

In [ ]:
# Separate generator for reproducible data.
data_generator = torch.Generator().manual_seed(123)

def make_toy_data(n_examples, generator):
    # Independently sample u and m uniformly from [-2, 2).
    inputs = 4 * torch.rand(n_examples, 2, generator=generator) - 2

    # Target depends only on m, the second coordinate.
    targets = 2 * inputs[:, 1:2]
    return inputs, targets


x_train, y_train = make_toy_data(1024, data_generator)
x_val, y_val = make_toy_data(256, data_generator)

print("Training shapes:", x_train.shape, y_train.shape)
print("Validation shapes:", x_val.shape, y_val.shape)

print("\nFirst five training inputs [u, m]:")
print(x_train[:5])

print("\nCorresponding targets 2m:")
print(y_train[:5])

assert x_train.shape == (1024, 2)
assert y_train.shape == (1024, 1)
assert x_val.shape == (256, 2)
assert y_val.shape == (256, 1)

torch.testing.assert_close(y_train, 2 * x_train[:, 1:2])
torch.testing.assert_close(y_val, 2 * x_val[:, 1:2])

Training shapes: torch.Size([1024, 2]) torch.Size([1024, 1])
Validation shapes: torch.Size([256, 2]) torch.Size([256, 1])

First five training inputs [u, m]:
tensor([[-0.8156,  0.0662],
        [-0.9933,  0.7542],
        [-1.7041,  1.4661],
        [-1.4537, -1.5901],
        [-1.2638,  0.9058]])

Corresponding targets 2m:
tensor([[ 0.1325],
        [ 1.5085],
        [ 2.9322],
        [-3.1802],
        [ 1.8116]])


In [ ]:
import copy

# Preserve the initialization for the later frozen-component variants.
initial_transcoder = copy.deepcopy(transcoder)
learned_transcoder = copy.deepcopy(initial_transcoder)

optimizer = torch.optim.Adam(
    learned_transcoder.parameters(),
    lr=cfg.lr,
)

shuffle_generator = torch.Generator().manual_seed(456)


def validation_metrics(model):
    model.eval()
    with torch.no_grad():
        _, features, _, mse, _ = toy_forward(model, x_val, y_val)
        mean_l0 = (features > 0).float().sum(dim=-1).mean()

    return mse.item(), mean_l0.item()


initial_mse, initial_l0 = validation_metrics(learned_transcoder)
print(f"Before training | Val MSE: {initial_mse:.4f} | L0: {initial_l0:.2f}")

n_epochs = 100

for epoch in range(1, n_epochs + 1):
    learned_transcoder.train()

    # Shuffle example order for this epoch.
    order = torch.randperm(
        len(x_train), generator=shuffle_generator
    )

    for start in range(0, len(x_train), cfg.train_batch_size):
        indices = order[start : start + cfg.train_batch_size]
        x_batch = x_train[indices]
        y_batch = y_train[indices]

        optimizer.zero_grad(set_to_none=True)

        _, _, loss, _, _ = toy_forward(
            learned_transcoder, x_batch, y_batch
        )
        assert torch.isfinite(loss).item()

        loss.backward()
        optimizer.step()

        # Keep each decoder row at unit norm.
        with torch.no_grad():
            weights = learned_transcoder.W_dec
            norms = weights.norm(dim=-1, keepdim=True)
            assert (norms > 0).all().item()
            weights.div_(norms)

    if epoch == 1 or epoch % 20 == 0:
        val_mse, val_l0 = validation_metrics(learned_transcoder)
        print(
            f"Epoch {epoch:3d} | "
            f"Val MSE: {val_mse:.4f} | "
            f"L0: {val_l0:.2f}"
        )

Before training | Val MSE: 16.8875 | L0: 4.10
Epoch   1 | Val MSE: 16.1821 | L0: 4.09
Epoch  20 | Val MSE: 7.0043 | L0: 3.71
Epoch  40 | Val MSE: 2.9673 | L0: 3.27
Epoch  60 | Val MSE: 1.3462 | L0: 2.98
Epoch  80 | Val MSE: 0.6565 | L0: 2.96
Epoch 100 | Val MSE: 0.3515 | L0: 2.99


| Code | What it does |
|---|---|
| `copy.deepcopy(...)` | Makes independent model copies, preserving the original initialization. |
| `optimizer.zero_grad(...)` | Clears gradients from the previous batch. |
| `toy_forward(...)` | Calculates predictions and MSE + L1 loss. |
| `loss.backward()` | Calculates gradients of that loss with respect to parameters. |
| `optimizer.step()` | Updates parameters using those gradients. |
| `weights.div_(norms)` | Restores unit-length decoder directions after the update. |

| Check | Example | Expected original response | Purpose |
|---|---|---:|---|
| **Identity edit** | `[1, 1] → [1, 1]` | `0` | Check that unchanged inputs give zero change. |
| **Change the relevant variable `m`** | `[1, 1] → [1, 3]` | `2 → 6`, so **+4** | Does the replacement capture a real effect? |
| **Change the irrelevant variable `u`** | `[1, 1] → [3, 1]` | `2 → 2`, so **0** | Does it invent an incorrect dependency on `u`? |
| **Decrease `m`** | `[1, 1] → [1, −1]` | `2 → −2`, so **−4** | Check responses in the opposite direction. |
| **Nonlinear ReLU test** | `m: −1 → 1` | `ReLU(m): 0 → 1`, so **+1** | Check effects when an edit crosses an activation boundary. |
| **Redundant OR pathways** | `[1,1] → [0,1]` versus `[1,1] → [0,0]` | **0** versus **−1** | Individual removals can hide effects that joint removal reveals. |

In [ ]:
def original(x):
    return 2 * x[..., 1:2]


def replacement(x):
    prediction, *_ = toy_forward(
        learned_transcoder, x, target=original(x)
    )
    return prediction


def evaluate_response(f, g, x, x_changed):
    with torch.no_grad():
        f_clean = f(x)
        f_changed = f(x_changed)
        g_clean = g(x)
        g_changed = g(x_changed)

        assert (
            f_clean.shape == f_changed.shape
            == g_clean.shape == g_changed.shape
        )

        delta_true = f_changed - f_clean
        delta_pred = g_changed - g_clean
        response_error = delta_pred - delta_true

        score = response_error.square().sum(dim=-1).mean()

    return {
        "delta_true": delta_true,
        "delta_pred": delta_pred,
        "response_error": response_error,
        "mean_response_squared_error": score,
    }


learned_transcoder.eval()

# Select three validation examples away from the range boundaries.
mask = (x_val.abs() <= 1.0).all(dim=-1)
x_base = x_val[mask][:3].clone()
assert len(x_base) == 3

x_identity = x_base.clone()

x_increase_m = x_base.clone()
x_increase_m[:, 1] += 0.5

x_increase_u = x_base.clone()
x_increase_u[:, 0] += 0.5

x_decrease_m = x_base.clone()
x_decrease_m[:, 1] -= 0.5

cases = {
    "Identity":   (x_identity,    0.0),
    "Increase m": (x_increase_m,  1.0),
    "Increase u": (x_increase_u,  0.0),
    "Decrease m": (x_decrease_m, -1.0),
}

intervention_results = {}

print("Base inputs [u, m]:\n", x_base)

for name, (edited_inputs, expected_change) in cases.items():
    result = evaluate_response(
        original, replacement, x_base, edited_inputs
    )
    intervention_results[name] = result

    # Verify the original function's known response.
    torch.testing.assert_close(
        result["delta_true"],
        torch.full_like(result["delta_true"], expected_change),
    )

    print(f"\n{name}")
    print("True changes:",
          result["delta_true"].flatten().tolist())
    print("Predicted changes:",
          result["delta_pred"].flatten().tolist())
    print("Mean squared response error:",
          result["mean_response_squared_error"].item())

# An unchanged input should produce no predicted change.
torch.testing.assert_close(
    intervention_results["Identity"]["delta_pred"],
    torch.zeros_like(intervention_results["Identity"]["delta_pred"]),
)

NameError: name 'learned_transcoder' is not defined

In [ ]:
learned_transcoder.eval()

mask = (x_val.abs() <= 1.5).all(dim=-1)
x_panel = x_val[mask].clone()

print("Number of examples:", len(x_panel))

# (coordinate to edit, amount to add)
edits = {
    "Increase m": (1,  0.5),
    "Decrease m": (1, -0.5),
    "Increase u": (0,  0.5),
    "Decrease u": (0, -0.5),
}

panel_results = {}

for name, (coordinate, amount) in edits.items():
    x_edited = x_panel.clone()
    x_edited[:, coordinate] += amount

    result = evaluate_response(
        original, replacement, x_panel, x_edited
    )
    panel_results[name] = result

    errors = result["response_error"]
    rmse = result["mean_response_squared_error"].sqrt().item()
    mean_abs_error = errors.abs().mean().item()
    mean_change = result["delta_pred"].mean().item()

    print(
        f"{name:12s} | "
        f"Mean predicted change: {mean_change:+.3f} | "
        f"Response RMSE: {rmse:.3f} | "
        f"Mean absolute error: {mean_abs_error:.3f}"
    )

Number of examples: 137
Increase m   | Mean predicted change: +0.901 | Response RMSE: 0.328 | Mean absolute error: 0.246
Decrease m   | Mean predicted change: -0.809 | Response RMSE: 0.429 | Mean absolute error: 0.325
Increase u   | Mean predicted change: +0.020 | Response RMSE: 0.129 | Mean absolute error: 0.112
Decrease u   | Mean predicted change: -0.006 | Response RMSE: 0.139 | Mean absolute error: 0.120


In [ ]:
# Keep the epoch-100 results for comparison.
before_mse, before_l0 = validation_metrics(learned_transcoder)
before_panel_results = panel_results.copy()

for epoch in range(completed_epochs + 1, completed_epochs + 101):
    learned_transcoder.train()

    order = torch.randperm(
        len(x_train), generator=shuffle_generator
    )

    for start in range(0, len(x_train), cfg.train_batch_size):
        indices = order[start : start + cfg.train_batch_size]

        optimizer.zero_grad(set_to_none=True)

        _, _, loss, _, _ = toy_forward(
            learned_transcoder,
            x_train[indices],
            y_train[indices],
        )
        assert torch.isfinite(loss).item()

        loss.backward()
        optimizer.step()

        with torch.no_grad():
            weights = learned_transcoder.W_dec
            norms = weights.norm(dim=-1, keepdim=True)
            assert (norms > 0).all().item()
            weights.div_(norms)

    if epoch % 20 == 0:
        val_mse, val_l0 = validation_metrics(learned_transcoder)
        with torch.no_grad():
          _, _, _, train_mse, _ = toy_forward(
            learned_transcoder, x_train, y_train
          )

        print(
            f"Epoch {epoch} | "
            f"Train MSE: {train_mse.item():.4f} | "
            f"Val MSE: {val_mse:.4f} | "
            f"L0: {val_l0:.2f}"
                )

# Repeat the exact same intervention panel.
learned_transcoder.eval()
after_panel_results = {}

print("\nResponse RMSE: before → after")

for name, (coordinate, amount) in edits.items():
    x_edited = x_panel.clone()
    x_edited[:, coordinate] += amount

    result = evaluate_response(
        original, replacement, x_panel, x_edited
    )
    after_panel_results[name] = result

    before = before_panel_results[name][
        "mean_response_squared_error"
    ].sqrt().item()

    after = result["mean_response_squared_error"].sqrt().item()

    print(f"{name:12s} | {before:.4f} → {after:.4f}")


completed_epochs += 100

Epoch 120 | Train MSE: 0.0505 | Val MSE: 0.0513 | L0: 3.69
Epoch 140 | Train MSE: 0.0411 | Val MSE: 0.0413 | L0: 3.76
Epoch 160 | Train MSE: 0.0306 | Val MSE: 0.0299 | L0: 3.73
Epoch 180 | Train MSE: 0.0197 | Val MSE: 0.0189 | L0: 3.66
Epoch 200 | Train MSE: 0.0115 | Val MSE: 0.0118 | L0: 3.68

Response RMSE: before → after
Increase m   | 0.3282 → 0.1232
Decrease m   | 0.4290 → 0.0878
Increase u   | 0.1293 → 0.0472
Decrease u   | 0.1391 → 0.0515


I ran the cell above twice like a dumbfuck
| Intervention RMSE | Epoch 100 | Epoch 200 | Epoch 300 |
|---|---:|---:|---:|
| Increase `m` | 0.3282 | 0.2244 | **0.1232** |
| Decrease `m` | 0.4290 | 0.2653 | **0.0878** |
| Increase `u` | 0.1293 | 0.0786 | **0.0472** |
| Decrease `u` | 0.1391 | 0.0815 | **0.0515** |


In [ ]:
completed_epochs = 300
panel_results = after_panel_results.copy()


In [ ]:
# Fresh data; do not use it for gradient updates.
test_generator = torch.Generator().manual_seed(789)
x_test, y_test = make_toy_data(1024, test_generator)

learned_transcoder.eval()

with torch.no_grad():
    _, features, _, test_mse, _ = toy_forward(
        learned_transcoder, x_test, y_test
    )
    test_l0 = (features > 0).float().sum(dim=-1).mean()

print(f"Test reconstruction MSE: {test_mse.item():.4f}")
print(f"Test average L0: {test_l0.item():.2f}")

# Keep edited inputs inside the training range.
mask = (x_test.abs() <= 1.5).all(dim=-1)
x_test_panel = x_test[mask].clone()

print("Test intervention examples:", len(x_test_panel))

test_results = {}

for name, (coordinate, amount) in edits.items():
    x_edited = x_test_panel.clone()
    x_edited[:, coordinate] += amount

    result = evaluate_response(
        original, replacement, x_test_panel, x_edited
    )
    test_results[name] = result

    rmse = result["mean_response_squared_error"].sqrt().item()
    print(f"{name:12s} | Response RMSE: {rmse:.4f}")

Test reconstruction MSE: 0.0105
Test average L0: 3.62
Test intervention examples: 569
Increase m   | Response RMSE: 0.1023
Decrease m   | Response RMSE: 0.0966
Increase u   | Response RMSE: 0.0461
Decrease u   | Response RMSE: 0.0483


This supports the correctness of our workflow; it isn’t yet evidence that learned features outperform random ones.


| Step | What we’ll do | Why |
|---|---|---|
| **1. Define a vector-output toy** | Build a small, fixed computation with known dependencies and multiple output coordinates. | Decoder directions can vary meaningfully, unlike our scalar `+1/−1` case. |
| **2. Prepare shared data and edits** | Generate training, validation, and reserved test sets. Define identity edits, relevant-variable edits, and irrelevant-variable edits. | Every variant faces the same task and interventions. |
| **3. Create three variants** | Fully learned; random encoder matrix frozen; random decoder matrix frozen. Start from copies of the same initialization. | Isolate the effect of learning each matrix. |
| **4. Verify freezing** | Specify which biases remain trainable, check optimizer parameters, and compare frozen matrices before and after training. | Ensure “frozen” really means unchanged, including during normalization. |
| **5. Train and monitor** | Use the same loss definition, data order, and initial training budget. Track training/validation MSE, L0, and learning curves. | Detect poor fitting or incomplete optimization before interpreting differences. |
| **6. Compare intervention responses** | Apply the shared edits and compare each replacement’s output changes with the original’s. | Measure which dependencies each model reproduces. |
| **7. Examine reconstruction–sparsity trade-offs** | Try a small range of L1 coefficients; look for configurations with overlapping validation reconstruction quality and report L0. | Avoid mistaking a reconstruction disadvantage for an intervention-specific failure. |
| **8. Freeze choices and test** | Evaluate selected configurations on reserved test data; repeat with additional initialization seeds. | Check whether the findings generalize and depend on initialization. |

In [ ]:
def original_vector(x):
    m1 = x[..., 1:2]
    m2 = x[..., 2:3]

    return torch.cat(
        [2 * m1, m1 + m2],
        dim=-1,
    )


# One example with three input coordinates.
x_vector_check = torch.tensor([[7.0, 1.0, 2.0]])

clean = original_vector(x_vector_check)
torch.testing.assert_close(
    clean, torch.tensor([[2.0, 3.0]])
)

# Each case: (input edit, expected output change).
checks = {
    "Identity": (
        [0.0, 0.0, 0.0],
        [0.0, 0.0],
    ),
    "Increase u": (
        [0.5, 0.0, 0.0],
        [0.0, 0.0],
    ),
    "Increase m1": (
        [0.0, 0.5, 0.0],
        [1.0, 0.5],
    ),
    "Increase m2": (
        [0.0, 0.0, 0.5],
        [0.0, 0.5],
    ),
    "Decrease m1": (
        [0.0, -0.5, 0.0],
        [-1.0, -0.5],
    ),
}

print("Clean output:", clean.tolist())

for name, (edit, expected_change) in checks.items():
    edited_input = x_vector_check + torch.tensor([edit])

    change = original_vector(edited_input) - clean

    torch.testing.assert_close(
        change, torch.tensor([expected_change])
    )

    print(f"{name}: {change.tolist()}")

print("All known-response checks passed.")

Clean output: [[2.0, 3.0]]
Identity: [[0.0, 0.0]]
Increase u: [[0.0, 0.0]]
Increase m1: [[1.0, 0.5]]
Increase m2: [[0.0, 0.5]]
Decrease m1: [[-1.0, -0.5]]
All known-response checks passed.


In [ ]:
def make_vector_data(n_examples, seed):
    generator = torch.Generator().manual_seed(seed)

    # Three independent coordinates, each in [-2, 2).
    inputs = 4 * torch.rand(
        n_examples, 3, generator=generator
    ) - 2

    targets = original_vector(inputs)
    return inputs, targets


x_vector_train, y_vector_train = make_vector_data(2048, seed=1001)
x_vector_val, y_vector_val = make_vector_data(512, seed=1002)

# Reserve a separate seed for the final test.
# Generate and evaluate that set only after choosing configurations.
vector_test_seed = 1003

print("Training:", x_vector_train.shape, y_vector_train.shape)
print("Validation:", x_vector_val.shape, y_vector_val.shape)

print("\nFirst three inputs [u, m1, m2]:")
print(x_vector_train[:3])

print("\nTheir targets [2m1, m1 + m2]:")
print(y_vector_train[:3])

assert x_vector_train.shape == (2048, 3)
assert y_vector_train.shape == (2048, 2)
assert x_vector_val.shape == (512, 3)
assert y_vector_val.shape == (512, 2)

# Check each target coordinate separately.
torch.testing.assert_close(
    y_vector_train[:, 0],
    2 * x_vector_train[:, 1],
)
torch.testing.assert_close(
    y_vector_train[:, 1],
    x_vector_train[:, 1] + x_vector_train[:, 2],
)

Training: torch.Size([2048, 3]) torch.Size([2048, 2])
Validation: torch.Size([512, 3]) torch.Size([512, 2])

First three inputs [u, m1, m2]:
tensor([[-0.4182, -1.3626,  1.4251],
        [-1.3476, -1.2345, -0.8515],
        [-1.1460,  0.5952,  1.6683]])

Their targets 1, m1 + m2]:
tensor([[-2.7252,  0.0625],
        [-2.4690, -2.0860],
        [ 1.1904,  2.2635]])


| Parameter | Fully learned | Frozen encoder matrix | Frozen decoder matrix |
|---|---|---|---|
| `W_enc` | Train | Freeze | Train |
| `W_dec` | Train | Train | Freeze |
| All biases | Train | Train | Train |

In [ ]:
import copy

vector_cfg = LanguageModelSAERunnerConfig(
    is_transcoder=True,
    d_in=3,
    d_out=2,
    expansion_factor=4,       # 3 × 4 = 12 features

    device="cpu",
    dtype=torch.float32,
    seed=2001,

    l1_coefficient=1e-3,
    lr=3e-4,
    train_batch_size=32,

    b_dec_init_method="zeros",
    feature_sampling_method=None,
    use_ghost_grads=False,
    is_sparse_connection=False,
    log_to_wandb=False,

    context_size=1,
    store_batch_size=32,
    n_batches_in_buffer=1,
    total_training_tokens=2048 * 100,
)

torch.manual_seed(vector_cfg.seed)
vector_initial = SparseAutoencoder(vector_cfg)

# Independent copies with identical initial parameter values.
vector_models = {
    "Learned": copy.deepcopy(vector_initial),
    "Frozen encoder": copy.deepcopy(vector_initial),
    "Frozen decoder": copy.deepcopy(vector_initial),
}


# Freeze the weight (Only works in the forward pass)
vector_models["Frozen encoder"].W_enc.requires_grad_(False)
vector_models["Frozen decoder"].W_dec.requires_grad_(False)

# Save exact copies for checking that frozen matrices stay unchanged.
frozen_snapshots = {
    "Frozen encoder":
        vector_models["Frozen encoder"].W_enc.detach().clone(),
    "Frozen decoder":
        vector_models["Frozen decoder"].W_dec.detach().clone(),
}

for name, model in vector_models.items():
    assert model.W_enc.shape == (3, 12)
    assert model.W_dec.shape == (12, 2)

    # Confirm all variants start with identical parameter values.
    for parameter_name, parameter in model.named_parameters():
        reference = dict(vector_initial.named_parameters())[parameter_name]
        assert torch.equal(parameter, reference)

    print(f"\n{name}")
    for parameter_name, parameter in model.named_parameters():
        status = "TRAIN" if parameter.requires_grad else "FROZEN"
        print(f"  {parameter_name:12s} {str(tuple(parameter.shape)):10s} {status}")

Run name: 12-L1-0.001-LR-0.0003-Tokens-2.048e+05
n_tokens_per_buffer (millions): 3.2e-05
Lower bound: n_contexts_per_buffer (millions): 3.2e-05
Total training steps: 6400
Total wandb updates: 640
n_tokens_per_feature_sampling_window (millions): 0.064
n_tokens_per_dead_feature_window (millions): 0.032
We will reset the sparsity calculation 3 times.
Number of tokens when resampling: 1024
Number tokens in sparsity calculation window: 6.40e+04

Learned
  W_enc        (3, 12)    TRAIN
  b_enc        (12,)      TRAIN
  W_dec        (12, 2)    TRAIN
  b_dec        (3,)       TRAIN
  b_dec_out    (2,)       TRAIN

Frozen encoder
  W_enc        (3, 12)    FROZEN
  b_enc        (12,)      TRAIN
  W_dec        (12, 2)    TRAIN
  b_dec        (3,)       TRAIN
  b_dec_out    (2,)       TRAIN

Frozen decoder
  W_enc        (3, 12)    TRAIN
  b_enc        (12,)      TRAIN
  W_dec        (12, 2)    FROZEN
  b_dec        (3,)       TRAIN
  b_dec_out    (2,)       TRAIN


In [ ]:
# Each optimizer receives only that model's trainable parameters.
vector_optimizers = {}

for name, model in vector_models.items():
    trainable_parameters = [
        p for p in model.parameters() if p.requires_grad
    ]

    vector_optimizers[name] = torch.optim.Adam(
        trainable_parameters,
        lr=vector_cfg.lr,
    )

    # Verify that the optimizer contains exactly those parameters.
    optimizer_ids = {
        id(p)
        for group in vector_optimizers[name].param_groups
        for p in group["params"]
    }
    expected_ids = {
        id(p) for p in model.parameters() if p.requires_grad
    }
    assert optimizer_ids == expected_ids

    count = sum(p.numel() for p in trainable_parameters)
    print(f"{name}: {count} trainable parameter values")


# Compare predictions before any training.
starting_predictions = {}

with torch.no_grad():
    for name, model in vector_models.items():
        model.eval()

        prediction, features, loss, mse, _ = toy_forward(
            model, x_vector_val, y_vector_val
        )

        assert prediction.shape == (512, 2)
        assert features.shape == (512, 12)
        assert torch.isfinite(loss).item()

        starting_predictions[name] = prediction.clone()

        print(f"{name} | Initial validation MSE: {mse.item():.4f}")

for prediction in starting_predictions.values():
    torch.testing.assert_close(
        prediction, starting_predictions["Learned"]
    )

# Initialize progress tracking for the upcoming training loop.
vector_completed_epochs = 0
vector_shuffle_generator = torch.Generator().manual_seed(3001)

print("All three starting predictions match.")

Learned: 77 trainable parameter values
Frozen encoder: 41 trainable parameter values
Frozen decoder: 53 trainable parameter values
Learned | Initial validation MSE: 6.1035
Frozen encoder | Initial validation MSE: 6.1035
Frozen decoder | Initial validation MSE: 6.1035
All three starting predictions match.


In [ ]:
def vector_metrics(model, inputs, targets):
    model.eval()
    with torch.no_grad():
        _, features, _, mse, _ = toy_forward(model, inputs, targets)
        l0 = (features > 0).float().sum(dim=-1).mean()
    return mse.item(), l0.item()


def check_frozen_matrices():
    assert torch.equal(
        vector_models["Frozen encoder"].W_enc,
        frozen_snapshots["Frozen encoder"],
    ), "Frozen encoder matrix changed!"

    assert torch.equal(
        vector_models["Frozen decoder"].W_dec,
        frozen_snapshots["Frozen decoder"],
    ), "Frozen decoder matrix changed!"


# Preserve logs if we later continue training.
if vector_completed_epochs == 0:
    vector_history = []

additional_epochs = 100

for _ in range(additional_epochs):
    for model in vector_models.values():
        model.train()

    # Generate one shared ordering per epoch.
    order = torch.randperm(
        len(x_vector_train),
        generator=vector_shuffle_generator,
    )

    for start in range(0, len(order), vector_cfg.train_batch_size):
        indices = order[start : start + vector_cfg.train_batch_size]
        x_batch = x_vector_train[indices]
        y_batch = y_vector_train[indices]

        # Every model receives the same batch.
        for name, model in vector_models.items():
            optimizer = vector_optimizers[name]
            optimizer.zero_grad(set_to_none=True)

            _, _, loss, _, _ = toy_forward(model, x_batch, y_batch)
            assert torch.isfinite(loss).item(), f"Invalid loss: {name}"

            loss.backward()
            optimizer.step()

            # Normalize only trainable decoder matrices.
            if model.W_dec.requires_grad:
                with torch.no_grad():
                    norms = model.W_dec.norm(dim=-1, keepdim=True)
                    assert (norms > 0).all().item()
                    model.W_dec.div_(norms)

    check_frozen_matrices()
    vector_completed_epochs += 1

    if vector_completed_epochs == 1 or vector_completed_epochs % 20 == 0:
        print(f"\nEpoch {vector_completed_epochs}")

        for name, model in vector_models.items():
            train_mse, _ = vector_metrics(
                model, x_vector_train, y_vector_train
            )
            val_mse, val_l0 = vector_metrics(
                model, x_vector_val, y_vector_val
            )

            vector_history.append({
                "epoch": vector_completed_epochs,
                "variant": name,
                "train_mse": train_mse,
                "val_mse": val_mse,
                "val_l0": val_l0,
            })

            print(
                f"{name:15s} | Train MSE: {train_mse:.4f} | "
                f"Val MSE: {val_mse:.4f} | L0: {val_l0:.2f}"
            )
            save_vector_checkpoints()

print("\nFrozen matrices are exactly unchanged.")


Epoch 120
Learned         | Train MSE: 0.0052 | Val MSE: 0.0052 | L0: 6.57
Frozen encoder  | Train MSE: 1.5452 | Val MSE: 1.3761 | L0: 4.87
Frozen decoder  | Train MSE: 0.0245 | Val MSE: 0.0237 | L0: 7.23

Epoch 140
Learned         | Train MSE: 0.0019 | Val MSE: 0.0019 | L0: 6.54
Frozen encoder  | Train MSE: 1.4916 | Val MSE: 1.3266 | L0: 4.84
Frozen decoder  | Train MSE: 0.0157 | Val MSE: 0.0158 | L0: 7.53

Epoch 160
Learned         | Train MSE: 0.0007 | Val MSE: 0.0007 | L0: 6.52
Frozen encoder  | Train MSE: 1.4507 | Val MSE: 1.2885 | L0: 4.83
Frozen decoder  | Train MSE: 0.0079 | Val MSE: 0.0082 | L0: 7.76

Epoch 180
Learned         | Train MSE: 0.0003 | Val MSE: 0.0004 | L0: 6.50
Frozen encoder  | Train MSE: 1.4163 | Val MSE: 1.2554 | L0: 4.86
Frozen decoder  | Train MSE: 0.0040 | Val MSE: 0.0042 | L0: 7.81

Epoch 200
Learned         | Train MSE: 0.0001 | Val MSE: 0.0002 | L0: 6.45
Frozen encoder  | Train MSE: 1.3871 | Val MSE: 1.2276 | L0: 4.91
Frozen decoder  | Train MSE: 0.0024

In [ ]:
from google.colab import drive
from pathlib import Path
import copy

drive.mount("/content/drive")

checkpoint_dir = Path(
    "/content/drive/MyDrive/transcoder_pilot/vector_run_seed2001"
)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Preserve best scores if this cell is rerun in the same session.
if "vector_best_mse" not in globals():
    vector_best_mse = {
        name: float("inf") for name in vector_models
    }


def save_vector_checkpoints():
    for name, model in vector_models.items():
        val_mse, val_l0 = vector_metrics(
            model, x_vector_val, y_vector_val
        )

        checkpoint = {
            "variant": name,
            "epoch": vector_completed_epochs,
            "model_state": copy.deepcopy(model.state_dict()),
            "optimizer_state": copy.deepcopy(
                vector_optimizers[name].state_dict()
            ),
            "val_mse": val_mse,
            "val_l0": val_l0,
            "config": copy.deepcopy(vars(model.cfg)),
            "requires_grad": {
                key: parameter.requires_grad
                for key, parameter in model.named_parameters()
            },
            "shuffle_rng_state": vector_shuffle_generator.get_state(),
            "torch_rng_state": torch.get_rng_state(),
            "frozen_snapshots": copy.deepcopy(frozen_snapshots),
            "data_seeds": {
                "train": 1001,
                "validation": 1002,
                "test": vector_test_seed,
            },
            "history": copy.deepcopy(vector_history),
            "objective": "ordinary coordinate-mean MSE + L1 activations",
        }

        filename = name.lower().replace(" ", "_")

        # Always save the current state for resuming.
        torch.save(
            checkpoint,
            checkpoint_dir / f"{filename}_latest.pt",
        )

        # Replace the best checkpoint only when validation MSE improves.
        if val_mse < vector_best_mse[name]:
            vector_best_mse[name] = val_mse
            torch.save(
                checkpoint,
                checkpoint_dir / f"{filename}_best.pt",
            )
            print(
                f"Saved best: {name} | "
                f"Epoch {vector_completed_epochs} | "
                f"Val MSE {val_mse:.6f}"
            )

    torch.save(vector_best_mse, checkpoint_dir / "best_scores.pt")


save_vector_checkpoints()

Mounted at /content/drive
Saved best: Learned | Epoch 200 | Val MSE 0.000193
Saved best: Frozen encoder | Epoch 200 | Val MSE 1.227606
Saved best: Frozen decoder | Epoch 200 | Val MSE 0.002448


In [ ]:
# Shared validation inputs; ±0.5 edits stay within [-2, 2].
mask = (x_vector_val.abs() <= 1.5).all(dim=-1)
vector_panel = x_vector_val[mask].clone()
assert len(vector_panel) > 0

vector_edits = {
    "Identity":    (0,  0.0),
    "Increase u":  (0,  0.5),
    "Decrease u":  (0, -0.5),
    "Increase m1": (1,  0.5),
    "Decrease m1": (1, -0.5),
    "Increase m2": (2,  0.5),
    "Decrease m2": (2, -0.5),
}

vector_response_results = {}

print("Intervention examples:", len(vector_panel))

for name, model in vector_models.items():
    model.eval()

    # Bind this particular model to its prediction function.
    def predict(inputs, current_model=model):
        prediction, *_ = toy_forward(
            current_model, inputs, original_vector(inputs)
        )
        return prediction

    print(f"\n{name}")

    for edit_name, (coordinate, amount) in vector_edits.items():
        edited = vector_panel.clone()
        edited[:, coordinate] += amount

        result = evaluate_response(
            original_vector, predict, vector_panel, edited
        )
        vector_response_results[(name, edit_name)] = result

        # Sum squared errors across BOTH output coordinates,
        # average across examples, then take the square root.
        vector_rmse = (
            result["mean_response_squared_error"].sqrt().item()
        )

        if edit_name == "Identity":
            torch.testing.assert_close(
                result["delta_pred"],
                torch.zeros_like(result["delta_pred"]),
            )

        print(f"{edit_name:12s} | Response vector RMSE: {vector_rmse:.4f}")


Intervention examples: 206

Learned
Identity     | Response vector RMSE: 0.0000
Increase u   | Response vector RMSE: 0.0138
Decrease u   | Response vector RMSE: 0.0152
Increase m1  | Response vector RMSE: 0.0220
Decrease m1  | Response vector RMSE: 0.0214
Increase m2  | Response vector RMSE: 0.0089
Decrease m2  | Response vector RMSE: 0.0104

Frozen encoder
Identity     | Response vector RMSE: 0.0000
Increase u   | Response vector RMSE: 0.4434
Decrease u   | Response vector RMSE: 0.4398
Increase m1  | Response vector RMSE: 0.4414
Decrease m1  | Response vector RMSE: 0.4400
Increase m2  | Response vector RMSE: 0.3735
Decrease m2  | Response vector RMSE: 0.3567

Frozen decoder
Identity     | Response vector RMSE: 0.0000
Increase u   | Response vector RMSE: 0.0188
Decrease u   | Response vector RMSE: 0.0222
Increase m1  | Response vector RMSE: 0.0713
Decrease m1  | Response vector RMSE: 0.0809
Increase m2  | Response vector RMSE: 0.0387
Decrease m2  | Response vector RMSE: 0.0408


**Our sequence today:**
1. **Restore the checkpoints and verify the results.** Load weights, optimizer states, freezing settings, and progress. Confirm that validation metrics match yesterday’s before training further.
2. **Diagnose the frozen encoder’s poor fit.** Give it a fixed additional training budget. If improvement stalls, investigate learning rate, feature count, and the restriction caused by fixed encoder weights combined with unit-length decoder rows.
3. **Find fair reconstruction comparisons.** Explore a small sparsity-penalty sweep and report reconstruction alongside L0. We need overlapping reconstruction quality before claiming an intervention advantage beyond reconstruction.
4. **Repeat the shared interventions on selected models.** Check relevant-variable responses and unwanted sensitivity to u. Reserve the untouched vector test set for after configuration choices.

**What we already established:** the frozen matrices stayed unchanged, the evaluator works, and the frozen encoder has large reconstruction and intervention errors. We haven’t established a separate causal-faithfulness failure.


In [ ]:
from google.colab import drive
from pathlib import Path
import torch

drive.mount("/content/drive")

checkpoint_dir = Path(
    "/content/drive/MyDrive/transcoder_pilot/vector_run_seed2001"
)

checkpoint_files = {
    "Learned": "learned_latest.pt",
    "Frozen encoder": "frozen_encoder_latest.pt",
    "Frozen decoder": "frozen_decoder_latest.pt",
}

loaded_checkpoints = {}

for name, filename in checkpoint_files.items():
    path = checkpoint_dir / filename
    assert path.exists(), f"Missing checkpoint: {path}"

    # These are the checkpoints you created yourself.
    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )
    loaded_checkpoints[name] = checkpoint

    assert checkpoint["variant"] == name

    print(
        f"{name:15s} | "
        f"Epoch: {checkpoint['epoch']} | "
        f"Val MSE: {checkpoint['val_mse']:.6f} | "
        f"L0: {checkpoint['val_l0']:.2f}"
    )

epochs = {
    checkpoint["epoch"]
    for checkpoint in loaded_checkpoints.values()
}
assert len(epochs) == 1, "Checkpoints come from different epochs!"

print("\nCheckpoint files loaded. Models are not restored yet.")

Mounted at /content/drive
Learned         | Epoch: 200 | Val MSE: 0.000193 | L0: 6.45
Frozen encoder  | Epoch: 200 | Val MSE: 1.227606 | L0: 4.91
Frozen decoder  | Epoch: 200 | Val MSE: 0.002448 | L0: 7.90

Checkpoint files loaded. Models are not restored yet.


In [ ]:
from types import SimpleNamespace
import copy

from sae_training.sparse_autoencoder import SparseAutoencoder

vector_models = {}
vector_optimizers = {}

for name, checkpoint in loaded_checkpoints.items():
    # Restore the saved configuration directly, including derived fields.
    restored_cfg = SimpleNamespace(**copy.deepcopy(checkpoint["config"]))
    restored_cfg.device = torch.device("cpu")

    model = SparseAutoencoder(restored_cfg)
    model.load_state_dict(checkpoint["model_state"], strict=True)

    # Freezing settings are not stored inside model.state_dict().
    for parameter_name, parameter in model.named_parameters():
        parameter.requires_grad_(
            checkpoint["requires_grad"][parameter_name]
        )

    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=restored_cfg.lr,
    )
    optimizer.load_state_dict(checkpoint["optimizer_state"])

    model.eval()
    vector_models[name] = model
    vector_optimizers[name] = optimizer

    # Verify every restored parameter/buffer against the saved values.
    for key, value in model.state_dict().items():
        assert torch.equal(value, checkpoint["model_state"][key])

    print(f"Restored: {name}")


# These shared states were saved at the same training boundary.
shared = loaded_checkpoints["Learned"]
vector_completed_epochs = shared["epoch"]
vector_cfg = vector_models["Learned"].cfg
vector_history = copy.deepcopy(shared["history"])
frozen_snapshots = copy.deepcopy(shared["frozen_snapshots"])
vector_test_seed = shared["data_seeds"]["test"]

vector_shuffle_generator = torch.Generator()
vector_shuffle_generator.set_state(shared["shuffle_rng_state"])

# Restore after constructing models, since construction consumes randomness.
torch.set_rng_state(shared["torch_rng_state"])

# Restore the best-checkpoint tracking scores.
vector_best_mse = torch.load(
    checkpoint_dir / "best_scores.pt",
    map_location="cpu",
    weights_only=True,
)

assert torch.equal(
    vector_models["Frozen encoder"].W_enc,
    frozen_snapshots["Frozen encoder"],
)
assert torch.equal(
    vector_models["Frozen decoder"].W_dec,
    frozen_snapshots["Frozen decoder"],
)

print(f"\nReady at epoch {vector_completed_epochs}.")
print("Saved weights restored exactly; frozen matrices verified.")

Restored: Learned
Restored: Frozen encoder
Restored: Frozen decoder

Ready at epoch 200.
Saved weights restored exactly; frozen matrices verified.


In [ ]:
# Restore the original toy function.
def original_vector(x):
    m1 = x[..., 1:2]
    m2 = x[..., 2:3]
    return torch.cat([2 * m1, m1 + m2], dim=-1)


def make_vector_data(n_examples, seed):
    generator = torch.Generator().manual_seed(seed)
    inputs = 4 * torch.rand(
        n_examples, 3, generator=generator
    ) - 2
    return inputs, original_vector(inputs)


# Restore our ordinary-MSE + L1 forward helper.
def toy_forward(model, x, target):
    x = x.to(device=model.W_enc.device, dtype=model.W_enc.dtype)
    target = target.to(device=x.device, dtype=x.dtype)

    centered = model.hook_sae_in(x - model.b_dec)
    scores = model.hook_hidden_pre(centered @ model.W_enc + model.b_enc)
    features = model.hook_hidden_post(torch.relu(scores))
    prediction = model.hook_sae_out(
        features @ model.W_dec + model.b_dec_out
    )

    assert prediction.shape == target.shape

    mse = (prediction - target).square().mean()
    l1 = model.l1_coefficient * features.abs().sum(dim=-1).mean()
    return prediction, features, mse + l1, mse, l1


def vector_metrics(model, inputs, targets):
    model.eval()
    with torch.no_grad():
        _, features, _, mse, _ = toy_forward(model, inputs, targets)
        l0 = (features > 0).float().sum(dim=-1).mean()
    return mse.item(), l0.item()


# Recreate the exact training and validation datasets.
seeds = loaded_checkpoints["Learned"]["data_seeds"]
x_vector_train, y_vector_train = make_vector_data(2048, seeds["train"])
x_vector_val, y_vector_val = make_vector_data(512, seeds["validation"])

for name, model in vector_models.items():
    mse, l0 = vector_metrics(model, x_vector_val, y_vector_val)
    checkpoint = loaded_checkpoints[name]

    torch.testing.assert_close(
        torch.tensor(mse),
        torch.tensor(checkpoint["val_mse"]),
        rtol=1e-5,
        atol=1e-7,
    )
    torch.testing.assert_close(
        torch.tensor(l0),
        torch.tensor(checkpoint["val_l0"]),
    )

    print(f"{name:15s} | Val MSE: {mse:.6f} | L0: {l0:.2f}")

print("\nRestoration verified. No training performed.")

Learned         | Val MSE: 0.000193 | L0: 6.45
Frozen encoder  | Val MSE: 1.227606 | L0: 4.91
Frozen decoder  | Val MSE: 0.002448 | L0: 7.90

Restoration verified. No training performed.


In [ ]:
import copy

# Create a diagnostic branch from the epoch-200 frozen encoder.
encoder_probe = copy.deepcopy(vector_models["Frozen encoder"])

probe_optimizer = torch.optim.Adam(
    [p for p in encoder_probe.parameters() if p.requires_grad],
    lr=encoder_probe.cfg.lr,
)
probe_optimizer.load_state_dict(copy.deepcopy(
    vector_optimizers["Frozen encoder"].state_dict()
))

probe_generator = torch.Generator()
probe_generator.set_state(
    loaded_checkpoints["Frozen encoder"]["shuffle_rng_state"]
)

probe_frozen_weights = encoder_probe.W_enc.detach().clone()
probe_history = []

start_epoch = loaded_checkpoints["Frozen encoder"]["epoch"]

for epoch in range(start_epoch + 1, start_epoch + 101):
    encoder_probe.train()

    order = torch.randperm(
        len(x_vector_train), generator=probe_generator
    )

    for start in range(0, len(order), encoder_probe.cfg.train_batch_size):
        indices = order[start : start + encoder_probe.cfg.train_batch_size]

        probe_optimizer.zero_grad(set_to_none=True)

        _, _, loss, _, _ = toy_forward(
            encoder_probe,
            x_vector_train[indices],
            y_vector_train[indices],
        )
        assert torch.isfinite(loss).item()

        loss.backward()
        probe_optimizer.step()

        # Decoder is trainable in this variant.
        with torch.no_grad():
            norms = encoder_probe.W_dec.norm(dim=-1, keepdim=True)
            assert (norms > 0).all().item()
            encoder_probe.W_dec.div_(norms)

    assert torch.equal(encoder_probe.W_enc, probe_frozen_weights)

    if epoch % 20 == 0:
        train_mse, _ = vector_metrics(
            encoder_probe, x_vector_train, y_vector_train
        )
        val_mse, l0 = vector_metrics(
            encoder_probe, x_vector_val, y_vector_val
        )

        probe_history.append({
            "epoch": epoch,
            "train_mse": train_mse,
            "val_mse": val_mse,
            "l0": l0,
        })

        print(
            f"Epoch {epoch} | Train MSE: {train_mse:.4f} | "
            f"Val MSE: {val_mse:.4f} | L0: {l0:.2f}"
        )

print("\nFrozen encoder matrix unchanged.")
print("Original three models remain at epoch 200.")

Epoch 220 | Train MSE: 1.3631 | Val MSE: 1.2036 | L0: 4.95
Epoch 240 | Train MSE: 1.3421 | Val MSE: 1.1827 | L0: 4.99
Epoch 260 | Train MSE: 1.3241 | Val MSE: 1.1651 | L0: 5.00
Epoch 280 | Train MSE: 1.3081 | Val MSE: 1.1513 | L0: 5.02
Epoch 300 | Train MSE: 1.2953 | Val MSE: 1.1403 | L0: 5.07

Frozen encoder matrix unchanged.
Original three models remain at epoch 200.


Slow improvement suggests that more epochs alone may be insufficient. An architectural bound can sometimes establish a limitation that learning curves cannot.

Let’s calculate an upper bound on how strongly the frozen-encoder model can respond. This check doesn’t train or change anything.
For an input-coordinate change of size \(\delta\), feature \(k\)’s activation can change by at most:



\[
|\delta| \cdot |W_{\text{enc}}[j,k]|
\]



Because each decoder row has length 1, summing those maximum contributions gives:



\[
\|\Delta \hat{y}\|_2 \;\leq\; |\delta| \sum_k |W_{\text{enc}}[j,k]|
\]

This is an optimistic ceiling: actual contributions may cancel, and some features may remain inactive.

In [ ]:
model = encoder_probe  # Frozen-encoder diagnostic at epoch 300.

with torch.no_grad():
    # Confirm the assumptions behind the bound.
    assert not model.W_enc.requires_grad
    decoder_norms = model.W_dec.norm(dim=-1)

    torch.testing.assert_close(
        decoder_norms,
        torch.ones_like(decoder_norms),
    )

    # Maximum response norm per unit change in each input coordinate.
    bounds = model.W_enc.abs().sum(dim=1)

    # True output changes for a +1 change in [u, m1, m2].
    required_vectors = torch.tensor([
        [0.0, 0.0],  # u
        [2.0, 1.0],  # m1
        [0.0, 1.0],  # m2
    ], device=bounds.device, dtype=bounds.dtype)

    required_norms = required_vectors.norm(dim=-1)

    edit_size = 0.5

    for j, name in enumerate(["u", "m1", "m2"]):
        ceiling = edit_size * bounds[j].item()
        required = edit_size * required_norms[j].item()

        print(
            f"{name:2s} | Response ceiling: {ceiling:.4f} | "
            f"Required response norm: {required:.4f}"
        )

        if ceiling + 1e-6 < required:
            print("   TOO SMALL: exact response is impossible "
                  "under these constraints.")
        else:
            print("   Inconclusive: this bound does not rule "
                  "out the required response.")

u  | Response ceiling: 2.5157 | Required response norm: 0.0000
   Inconclusive: this bound does not rule out the required response.
m1 | Response ceiling: 1.9281 | Required response norm: 1.1180
   Inconclusive: this bound does not rule out the required response.
m2 | Response ceiling: 2.1342 | Required response norm: 0.5000
   Inconclusive: this bound does not rule out the required response.


For Unit decoder, mean and maximum row norms should remain approximately 1. For Free decoder, they can change.
What we’re testing: does allowing the decoder to amplify or shrink feature contributions improve fitting compared with equally long continued training under the original constraint?
A better result for Free decoder would support the normalization constraint contributing to the poor fit. It would not prove that random features are equally sparse or equally useful: decoder scaling changes the relationship between activation magnitude and output contribution.

In [ ]:

import copy

# Both branches start from the same epoch-300 model.
decoder_probes = {
    "Unit decoder": copy.deepcopy(encoder_probe),
    "Free decoder": copy.deepcopy(encoder_probe),
}

decoder_probe_optimizers = {}

for name, model in decoder_probes.items():
    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=model.cfg.lr,
    )
    optimizer.load_state_dict(
        copy.deepcopy(probe_optimizer.state_dict())
    )
    decoder_probe_optimizers[name] = optimizer

# Continue from the same shuffle-generator state.
diagnostic_generator = torch.Generator()
diagnostic_generator.set_state(probe_generator.get_state())

fixed_encoder = encoder_probe.W_enc.detach().clone()
decoder_probe_history = []

for extra_epoch in range(1, 101):
    for model in decoder_probes.values():
        model.train()

    order = torch.randperm(
        len(x_vector_train), generator=diagnostic_generator
    )

    for start in range(0, len(order), vector_cfg.train_batch_size):
        indices = order[start : start + vector_cfg.train_batch_size]
        inputs = x_vector_train[indices]
        targets = y_vector_train[indices]

        for name, model in decoder_probes.items():
            optimizer = decoder_probe_optimizers[name]
            optimizer.zero_grad(set_to_none=True)

            _, _, loss, _, _ = toy_forward(model, inputs, targets)
            assert torch.isfinite(loss).item()

            loss.backward()
            optimizer.step()

            # This is the only training-rule difference.
            if name == "Unit decoder":
                with torch.no_grad():
                    norms = model.W_dec.norm(dim=-1, keepdim=True)
                    assert (norms > 0).all().item()
                    model.W_dec.div_(norms)

    for model in decoder_probes.values():
        assert torch.equal(model.W_enc, fixed_encoder)

    if extra_epoch == 1 or extra_epoch % 20 == 0:
        print(f"\nAdditional epoch {extra_epoch}")

        for name, model in decoder_probes.items():
            train_mse, _ = vector_metrics(
                model, x_vector_train, y_vector_train
            )
            val_mse, l0 = vector_metrics(
                model, x_vector_val, y_vector_val
            )

            with torch.no_grad():
                norms = model.W_dec.norm(dim=-1)
                mean_norm = norms.mean().item()
                max_norm = norms.max().item()

            decoder_probe_history.append({
                "extra_epoch": extra_epoch,
                "branch": name,
                "train_mse": train_mse,
                "val_mse": val_mse,
                "l0": l0,
                "mean_decoder_norm": mean_norm,
                "max_decoder_norm": max_norm,
            })

            print(
                f"{name:12s} | Train MSE: {train_mse:.4f} | "
                f"Val MSE: {val_mse:.4f} | L0: {l0:.2f} | "
                f"Decoder norm mean/max: "
                f"{mean_norm:.2f}/{max_norm:.2f}"
            )

print("\nBoth encoder matrices stayed exactly unchanged.")


Additional epoch 1
Unit decoder | Train MSE: 1.2946 | Val MSE: 1.1398 | L0: 5.08 | Decoder norm mean/max: 1.00/1.00
Free decoder | Train MSE: 1.2795 | Val MSE: 1.1278 | L0: 5.08 | Decoder norm mean/max: 1.00/1.02

Additional epoch 20
Unit decoder | Train MSE: 1.2817 | Val MSE: 1.1325 | L0: 5.11 | Decoder norm mean/max: 1.00/1.00
Free decoder | Train MSE: 1.0730 | Val MSE: 0.9633 | L0: 5.11 | Decoder norm mean/max: 1.01/1.34

Additional epoch 40
Unit decoder | Train MSE: 1.2693 | Val MSE: 1.1274 | L0: 5.17 | Decoder norm mean/max: 1.00/1.00
Free decoder | Train MSE: 0.9103 | Val MSE: 0.8258 | L0: 5.12 | Decoder norm mean/max: 1.02/1.63

Additional epoch 60
Unit decoder | Train MSE: 1.2543 | Val MSE: 1.1242 | L0: 5.23 | Decoder norm mean/max: 1.00/1.00
Free decoder | Train MSE: 0.7709 | Val MSE: 0.7030 | L0: 5.13 | Decoder norm mean/max: 1.04/1.91

Additional epoch 80
Unit decoder | Train MSE: 1.2392 | Val MSE: 1.1237 | L0: 5.31 | Decoder norm mean/max: 1.00/1.00
Free decoder | Train MS

In [ ]:
for extra_epoch in range(101, 201):
    for model in decoder_probes.values():
        model.train()

    order = torch.randperm(
        len(x_vector_train), generator=diagnostic_generator
    )

    for start in range(0, len(order), vector_cfg.train_batch_size):
        indices = order[start : start + vector_cfg.train_batch_size]
        inputs = x_vector_train[indices]
        targets = y_vector_train[indices]

        for name, model in decoder_probes.items():
            optimizer = decoder_probe_optimizers[name]
            optimizer.zero_grad(set_to_none=True)

            _, _, loss, _, _ = toy_forward(model, inputs, targets)
            assert torch.isfinite(loss).item()

            loss.backward()
            optimizer.step()

            # This is the only training-rule difference.
            if name == "Unit decoder":
                with torch.no_grad():
                    norms = model.W_dec.norm(dim=-1, keepdim=True)
                    assert (norms > 0).all().item()
                    model.W_dec.div_(norms)

    for model in decoder_probes.values():
        assert torch.equal(model.W_enc, fixed_encoder)

    if extra_epoch == 1 or extra_epoch % 20 == 0:
        print(f"\nAdditional epoch {extra_epoch}")

        for name, model in decoder_probes.items():
            train_mse, _ = vector_metrics(
                model, x_vector_train, y_vector_train
            )
            val_mse, l0 = vector_metrics(
                model, x_vector_val, y_vector_val
            )

            with torch.no_grad():
                norms = model.W_dec.norm(dim=-1)
                mean_norm = norms.mean().item()
                max_norm = norms.max().item()

            decoder_probe_history.append({
                "extra_epoch": extra_epoch,
                "branch": name,
                "train_mse": train_mse,
                "val_mse": val_mse,
                "l0": l0,
                "mean_decoder_norm": mean_norm,
                "max_decoder_norm": max_norm,
            })

            print(
                f"{name:12s} | Train MSE: {train_mse:.4f} | "
                f"Val MSE: {val_mse:.4f} | L0: {l0:.2f} | "
                f"Decoder norm mean/max: "
                f"{mean_norm:.2f}/{max_norm:.2f}"
            )

print("\nBoth encoder matrices stayed exactly unchanged.")


Additional epoch 120
Unit decoder | Train MSE: 1.2159 | Val MSE: 1.1058 | L0: 5.46 | Decoder norm mean/max: 1.00/1.00
Free decoder | Train MSE: 0.4379 | Val MSE: 0.4005 | L0: 5.16 | Decoder norm mean/max: 1.15/2.63

Additional epoch 140
Unit decoder | Train MSE: 1.2077 | Val MSE: 1.0980 | L0: 5.51 | Decoder norm mean/max: 1.00/1.00
Free decoder | Train MSE: 0.3187 | Val MSE: 0.2830 | L0: 5.36 | Decoder norm mean/max: 1.23/2.85

Additional epoch 160
Unit decoder | Train MSE: 1.2007 | Val MSE: 1.0898 | L0: 5.56 | Decoder norm mean/max: 1.00/1.00
Free decoder | Train MSE: 0.1334 | Val MSE: 0.1253 | L0: 5.91 | Decoder norm mean/max: 1.35/3.09

Additional epoch 180
Unit decoder | Train MSE: 1.1945 | Val MSE: 1.0823 | L0: 5.60 | Decoder norm mean/max: 1.00/1.00
Free decoder | Train MSE: 0.0575 | Val MSE: 0.0560 | L0: 6.19 | Decoder norm mean/max: 1.37/3.23

Additional epoch 200
Unit decoder | Train MSE: 1.1896 | Val MSE: 1.0761 | L0: 5.64 | Decoder norm mean/max: 1.00/1.00
Free decoder | Tr

***This strengthens the evidence that unit-length decoder normalization was hindering the frozen encoder’s fit***

That tells us something important: this frozen random encoder can support much better reconstruction than the original constrained run achieved. Its earlier poor fit wasn’t solely a consequence of having random encoder directions.

Two qualifications:
- The free branch now uses more active features, as well as variable decoder lengths. Those are outcomes of relaxing normalization.
- This experiment doesn’t distinguish a restriction on representable functions from an optimization difficulty caused by normalization. It demonstrates that removing normalization improves fitting under this training procedure

In [ ]:
# Same validation panel as before.
mask = (x_vector_val.abs() <= 1.5).all(dim=-1)
diagnostic_panel = x_vector_val[mask].clone()

diagnostic_edits = {
    "Identity":    (0,  0.0),
    "Increase u":  (0,  0.5),
    "Decrease u":  (0, -0.5),
    "Increase m1": (1,  0.5),
    "Decrease m1": (1, -0.5),
    "Increase m2": (2,  0.5),
    "Decrease m2": (2, -0.5),
}

diagnostic_responses = {}

print("Intervention examples:", len(diagnostic_panel))

with torch.no_grad():
    true_clean = original_vector(diagnostic_panel)

    for name, model in decoder_probes.items():
        model.eval()
        pred_clean, *_ = toy_forward(
            model, diagnostic_panel, true_clean
        )

        print(f"\n{name}")

        for edit_name, (coordinate, amount) in diagnostic_edits.items():
            edited = diagnostic_panel.clone()
            edited[:, coordinate] += amount

            true_edited = original_vector(edited)
            pred_edited, *_ = toy_forward(model, edited, true_edited)

            delta_true = true_edited - true_clean
            delta_pred = pred_edited - pred_clean
            error = delta_pred - delta_true

            vector_rmse = error.square().sum(dim=-1).mean().sqrt()

            diagnostic_responses[(name, edit_name)] = {
                "delta_true": delta_true.clone(),
                "delta_pred": delta_pred.clone(),
                "response_error": error.clone(),
                "vector_rmse": vector_rmse.item(),
            }

            if edit_name == "Identity":
                torch.testing.assert_close(
                    delta_pred, torch.zeros_like(delta_pred)
                )

            print(
                f"{edit_name:12s} | "
                f"Response vector RMSE: {vector_rmse.item():.4f}"
            )

Intervention examples: 206

Unit decoder
Identity     | Response vector RMSE: 0.0000
Increase u   | Response vector RMSE: 0.4862
Decrease u   | Response vector RMSE: 0.4680
Increase m1  | Response vector RMSE: 0.4092
Decrease m1  | Response vector RMSE: 0.4165
Increase m2  | Response vector RMSE: 0.1957
Decrease m2  | Response vector RMSE: 0.2329

Free decoder
Identity     | Response vector RMSE: 0.0000
Increase u   | Response vector RMSE: 0.1653
Decrease u   | Response vector RMSE: 0.1930
Increase m1  | Response vector RMSE: 0.0920
Decrease m1  | Response vector RMSE: 0.0715
Increase m2  | Response vector RMSE: 0.1809
Decrease m2  | Response vector RMSE: 0.1183


In [ ]:
import copy

diagnostic_path = checkpoint_dir / "decoder_norm_diagnostic_extra200.pt"

torch.save({
    "base_epoch": 300,
    "additional_epochs": 200,
    "models": {
        name: {
            "state_dict": copy.deepcopy(model.state_dict()),
            "config": copy.deepcopy(vars(model.cfg)),
            "requires_grad": {
                key: p.requires_grad
                for key, p in model.named_parameters()
            },
            "optimizer_state": copy.deepcopy(
                decoder_probe_optimizers[name].state_dict()
            ),
            "normalize_decoder": name == "Unit decoder",
        }
        for name, model in decoder_probes.items()
    },
    "shuffle_rng_state": diagnostic_generator.get_state(),
    "torch_rng_state": torch.get_rng_state(),
    "fixed_encoder": fixed_encoder.clone(),
    "history": copy.deepcopy(decoder_probe_history),
    "responses": diagnostic_responses,
    "data_seeds": loaded_checkpoints["Learned"]["data_seeds"],
    "train_examples": len(x_vector_train),
    "validation_examples": len(x_vector_val),
    "objective": "ordinary coordinate-mean MSE + L1 activations",
}, diagnostic_path)

print("Saved:", diagnostic_path)

Saved: /content/drive/MyDrive/transcoder_pilot/vector_run_seed2001/decoder_norm_diagnostic_extra200.pt


Two weaknesses remain:
- Unwanted dependence on u: its true effect is zero, but the free decoder still changes its output.
- Increasing m2: the error barely improves, despite much better overall reconstruction.

In [ ]:
for branch in ["Unit decoder", "Free decoder"]:
    print(f"\n{branch}")

    for edit_name in diagnostic_edits:
        if edit_name == "Identity":
            continue

        result = diagnostic_responses[(branch, edit_name)]

        # Shape of each tensor: [206 examples, 2 outputs].
        errors = result["response_error"]

        # Average over examples, keeping the two outputs separate.
        rmse_per_output = errors.square().mean(dim=0).sqrt()
        mean_prediction = result["delta_pred"].mean(dim=0)
        mean_true = result["delta_true"].mean(dim=0)
        mean_error = errors.mean(dim=0)

        print(f"\n  {edit_name}")
        print(
            f"    True mean change:      "
            f"[{mean_true[0].item():+.3f}, {mean_true[1].item():+.3f}]"
        )
        print(
            f"    Predicted mean change: "
            f"[{mean_prediction[0].item():+.3f}, "
            f"{mean_prediction[1].item():+.3f}]"
        )
        print(
            f"    Mean signed error:     "
            f"[{mean_error[0].item():+.3f}, {mean_error[1].item():+.3f}]"
        )
        print(
            f"    RMSE per output:       "
            f"[{rmse_per_output[0].item():.3f}, "
            f"{rmse_per_output[1].item():.3f}]"
        )


Unit decoder

  Increase u
    True mean change:      [+0.000, +0.000]
    Predicted mean change: [+0.393, +0.278]
    Mean signed error:     [+0.393, +0.278]
    RMSE per output:       [0.395, 0.283]

  Decrease u
    True mean change:      [+0.000, +0.000]
    Predicted mean change: [-0.378, -0.262]
    Mean signed error:     [-0.378, -0.262]
    RMSE per output:       [0.381, 0.272]

  Increase m1
    True mean change:      [+1.000, +0.500]
    Predicted mean change: [+0.670, +0.262]
    Mean signed error:     [-0.330, -0.238]
    RMSE per output:       [0.332, 0.239]

  Decrease m1
    True mean change:      [-1.000, -0.500]
    Predicted mean change: [-0.661, -0.262]
    Mean signed error:     [+0.339, +0.238]
    RMSE per output:       [0.341, 0.239]

  Increase m2
    True mean change:      [+0.000, +0.500]
    Predicted mean change: [-0.151, +0.406]
    Mean signed error:     [-0.151, -0.094]
    RMSE per output:       [0.158, 0.116]

  Decrease m2
    True mean change:      [

**Our diagnosis is now more precise:** relaxing decoder normalization greatly reduces systematic response errors, while input-dependent errors remain. We haven’t yet established whether those remaining errors come from ReLU activation boundaries, incomplete training, or limited fixed features.

In [ ]:
model = decoder_probes["Free decoder"]
model.eval()

feature_switch_results = {}

with torch.no_grad():
    true_clean = original_vector(diagnostic_panel)
    pred_clean, features_clean, *_ = toy_forward(
        model, diagnostic_panel, true_clean
    )
    active_clean = features_clean > 0

    for edit_name, (coordinate, amount) in diagnostic_edits.items():
        if edit_name == "Identity":
            continue

        edited = diagnostic_panel.clone()
        edited[:, coordinate] += amount

        true_edited = original_vector(edited)
        pred_edited, features_edited, *_ = toy_forward(
            model, edited, true_edited
        )

        active_edited = features_edited > 0

        # Count feature on/off switches separately for each example.
        switches = (active_clean != active_edited).sum(dim=-1)

        delta_true = true_edited - true_clean
        delta_pred = pred_edited - pred_clean
        response_error = delta_pred - delta_true

        # One squared response-vector error per example.
        squared_errors = response_error.square().sum(dim=-1)

        feature_switch_results[edit_name] = {
            "switch_count": switches.clone(),
            "response_error": response_error.clone(),
            "squared_error": squared_errors.clone(),
        }

        print(f"\n{edit_name}")
        print(f"Average feature switches: {switches.float().mean().item():.2f}")

        for label, mask in {
            "No switches": switches == 0,
            "At least one switch": switches > 0,
        }.items():
            count = mask.sum().item()

            if count == 0:
                print(f"  {label}: 0 examples")
                continue

            rmse = squared_errors[mask].mean().sqrt().item()

            print(
                f"  {label}: {count} examples | "
                f"Response vector RMSE: {rmse:.4f}"
            )


Increase u
Average feature switches: 0.32
  No switches: 145 examples | Response vector RMSE: 0.1705
  At least one switch: 61 examples | Response vector RMSE: 0.1522

Decrease u
Average feature switches: 0.33
  No switches: 144 examples | Response vector RMSE: 0.2087
  At least one switch: 62 examples | Response vector RMSE: 0.1500

Increase m1
Average feature switches: 0.23
  No switches: 166 examples | Response vector RMSE: 0.0685
  At least one switch: 40 examples | Response vector RMSE: 0.1553

Decrease m1
Average feature switches: 0.17
  No switches: 176 examples | Response vector RMSE: 0.0496
  At least one switch: 30 examples | Response vector RMSE: 0.1437

Increase m2
Average feature switches: 0.30
  No switches: 149 examples | Response vector RMSE: 0.1780
  At least one switch: 57 examples | Response vector RMSE: 0.1883

Decrease m2
Average feature switches: 0.35
  No switches: 147 examples | Response vector RMSE: 0.1152
  At least one switch: 59 examples | Response vector R

In [ ]:
model = decoder_probes["Free decoder"]
model.eval()

with torch.no_grad():
    _, features, *_ = toy_forward(
        model,
        diagnostic_panel,
        original_vector(diagnostic_panel),
    )

    active = (features > 0).to(model.W_enc.dtype)

    # Shape: [examples, input coordinates, output coordinates].
    local_slopes = torch.einsum(
        "jk,bk,ko->bjo",
        model.W_enc,
        active,
        model.W_dec,
    )

    # Rows: u, m1, m2. Columns: output 1, output 2.
    expected_slopes = local_slopes.new_tensor([
        [0.0, 0.0],
        [2.0, 1.0],
        [0.0, 1.0],
    ])

    slope_errors = local_slopes - expected_slopes
    mean_slopes = local_slopes.mean(dim=0)
    slope_rmse = slope_errors.square().mean(dim=0).sqrt()

    for j, name in enumerate(["u", "m1", "m2"]):
        print(f"\nInput: {name}")
        print("Expected slopes:", expected_slopes[j].tolist())
        print("Mean slopes:",
              [round(v, 4) for v in mean_slopes[j].tolist()])
        print("Slope RMSE per output:",
              [round(v, 4) for v in slope_rmse[j].tolist()])


Input: u
Expected slopes: [0.0, 0.0]
Mean slopes: [0.0152, 0.016]
Slope RMSE per output: [0.3395, 0.1989]

Input: m1
Expected slopes: [2.0, 1.0]
Mean slopes: [2.031, 1.0271]
Slope RMSE per output: [0.1468, 0.1054]

Input: m2
Expected slopes: [0.0, 1.0]
Mean slopes: [-0.0314, 0.9937]
Slope RMSE per output: [0.2559, 0.216]


In [7]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

checkpoint_dir = Path(
    "/content/drive/MyDrive/transcoder_pilot/vector_run_seed2001"
)

if not checkpoint_dir.exists():
    print("Checkpoint folder not found:", checkpoint_dir)
else:
    files = sorted(checkpoint_dir.glob("*.pt"))
    for path in files:
        print(f"{path.name} | {path.stat().st_size / 1e6:.2f} MB")

    diagnostic_path = (
        checkpoint_dir / "decoder_norm_diagnostic_extra200.pt"
    )
    print("\nDiagnostic checkpoint found:", diagnostic_path.exists())

Mounted at /content/drive
best_scores.pt | 0.00 MB
decoder_norm_diagnostic_extra200.pt | 0.11 MB
frozen_decoder_best.pt | 0.02 MB
frozen_decoder_latest.pt | 0.02 MB
frozen_encoder_best.pt | 0.02 MB
frozen_encoder_latest.pt | 0.02 MB
learned_best.pt | 0.02 MB
learned_latest.pt | 0.02 MB

Diagnostic checkpoint found: True


In [8]:
import torch

# weights_only=False because this is our own trusted checkpoint
# and it includes configuration and optimizer metadata.
diagnostic_checkpoint = torch.load(
    diagnostic_path,
    map_location="cpu",
    weights_only=False,
)

print("Checkpoint keys:", list(diagnostic_checkpoint.keys()))
print("Base epoch:", diagnostic_checkpoint.get("base_epoch"))
print("Additional epochs:", diagnostic_checkpoint.get("additional_epochs"))

for name, saved in diagnostic_checkpoint["models"].items():
    print(f"\n{name}")
    print("Saved fields:", list(saved.keys()))

    state = saved["state_dict"]
    for parameter in ["W_enc", "W_dec", "b_enc", "b_dec", "b_dec_out"]:
        print(f"  {parameter}: {tuple(state[parameter].shape)}")

Checkpoint keys: ['base_epoch', 'additional_epochs', 'models', 'shuffle_rng_state', 'torch_rng_state', 'fixed_encoder', 'history', 'responses', 'data_seeds', 'train_examples', 'validation_examples', 'objective']
Base epoch: 300
Additional epochs: 200

Unit decoder
Saved fields: ['state_dict', 'config', 'requires_grad', 'optimizer_state', 'normalize_decoder']
  W_enc: (3, 12)
  W_dec: (12, 2)
  b_enc: (12,)
  b_dec: (3,)
  b_dec_out: (2,)

Free decoder
Saved fields: ['state_dict', 'config', 'requires_grad', 'optimizer_state', 'normalize_decoder']
  W_enc: (3, 12)
  W_dec: (12, 2)
  b_enc: (12,)
  b_dec: (3,)
  b_dec_out: (2,)


In [9]:
def original_vector(x):
    m1 = x[:, 1:2]
    m2 = x[:, 2:3]
    return torch.cat([2 * m1, m1 + m2], dim=-1)


def make_vector_data(n_examples, seed):
    generator = torch.Generator().manual_seed(seed)
    x = 4 * torch.rand(n_examples, 3, generator=generator) - 2
    return x, original_vector(x)


# Same data seeds used in our original experiment.
x_vector_train, y_vector_train = make_vector_data(2048, seed=1001)
x_vector_val, y_vector_val = make_vector_data(512, seed=1002)


def forward_from_state(state, x):
    x = x.to(state["W_enc"])
    features = torch.relu(
        (x - state["b_dec"]) @ state["W_enc"] + state["b_enc"]
    )
    predictions = features @ state["W_dec"] + state["b_dec_out"]
    return predictions, features


with torch.no_grad():
    for name, saved in diagnostic_checkpoint["models"].items():
        predictions, features = forward_from_state(
            saved["state_dict"], x_vector_val
        )

        mse = (predictions - y_vector_val.to(predictions)).square().mean()
        l0 = (features > 0).float().sum(dim=-1).mean()

        print(f"{name:15s} | Val MSE: {mse.item():.4f} | L0: {l0.item():.2f}")

Unit decoder    | Val MSE: 1.0761 | L0: 5.64
Free decoder    | Val MSE: 0.0342 | L0: 6.30


In [10]:
import copy

free_state = diagnostic_checkpoint["models"]["Free decoder"]["state_dict"]

# Work on a separate copy in float64 for the least-squares calculation.
ls_state = {
    name: tensor.detach().clone().to(dtype=torch.float64)
    for name, tensor in free_state.items()
}

with torch.no_grad():
    _, train_features = forward_from_state(ls_state, x_vector_train)

    # The extra column of ones lets us fit the output bias too.
    design = torch.cat(
        [
            train_features,
            torch.ones(
                len(train_features), 1,
                dtype=train_features.dtype,
                device=train_features.device,
            ),
        ],
        dim=1,
    )
    targets = y_vector_train.to(design)

    fit = torch.linalg.lstsq(design, targets, driver="gelsd")

    ls_state["W_dec"] = fit.solution[:-1].clone()
    ls_state["b_dec_out"] = fit.solution[-1].clone()

    print(f"Design matrix rank: {fit.rank.item()} / {design.shape[1]}")

    for name, state in [
        ("Before least squares", free_state),
        ("After least squares", ls_state),
    ]:
        train_pred, _ = forward_from_state(state, x_vector_train)
        val_pred, val_features = forward_from_state(state, x_vector_val)

        train_mse = (
            train_pred - y_vector_train.to(train_pred)
        ).square().mean()

        val_mse = (
            val_pred - y_vector_val.to(val_pred)
        ).square().mean()

        l0 = (val_features > 0).float().sum(-1).mean()

        print(
            f"{name:22s} | Train MSE: {train_mse.item():.6f}"
            f" | Val MSE: {val_mse.item():.6f}"
            f" | L0: {l0.item():.2f}"
        )

# Verify that all feature-producing parameters stayed unchanged.
for name in ["W_enc", "b_enc", "b_dec"]:
    assert torch.equal(ls_state[name], free_state[name].double())

print("\nFeature-producing parameters unchanged.")

Design matrix rank: 11 / 13
Before least squares   | Train MSE: 0.036788 | Val MSE: 0.034247 | L0: 6.30
After least squares    | Train MSE: 0.000000 | Val MSE: 0.000000 | L0: 6.30

Feature-producing parameters unchanged.


In [11]:
panel = x_vector_val[
    (x_vector_val.abs() <= 1.5).all(dim=-1)
].clone()

edits = {
    "Identity":    (0,  0.0),
    "Increase u":  (0,  0.5),
    "Decrease u":  (0, -0.5),
    "Increase m1": (1,  0.5),
    "Decrease m1": (1, -0.5),
    "Increase m2": (2,  0.5),
    "Decrease m2": (2, -0.5),
}

print(f"Intervention examples: {len(panel)}")

with torch.no_grad():
    for name, state in [
        ("Before least squares", free_state),
        ("After least squares", ls_state),
    ]:
        base_pred, _ = forward_from_state(state, panel)
        base_true = original_vector(panel)

        print(f"\n{name}")

        for edit, (coordinate, amount) in edits.items():
            changed = panel.clone()
            changed[:, coordinate] += amount

            changed_pred, _ = forward_from_state(state, changed)

            true_response = original_vector(changed) - base_true
            predicted_response = changed_pred - base_pred

            error = predicted_response - true_response.to(predicted_response)
            rmse = error.square().sum(dim=-1).mean().sqrt()

            print(f"{edit:12s} | Response vector RMSE: {rmse.item():.6e}")


Intervention examples: 206

Before least squares
Identity     | Response vector RMSE: 0.000000e+00
Increase u   | Response vector RMSE: 1.652788e-01
Decrease u   | Response vector RMSE: 1.929527e-01
Increase m1  | Response vector RMSE: 9.201767e-02
Decrease m1  | Response vector RMSE: 7.146129e-02
Increase m2  | Response vector RMSE: 1.808940e-01
Decrease m2  | Response vector RMSE: 1.183292e-01

After least squares
Identity     | Response vector RMSE: 0.000000e+00
Increase u   | Response vector RMSE: 1.435670e-14
Decrease u   | Response vector RMSE: 1.386164e-14
Increase m1  | Response vector RMSE: 1.223706e-14
Decrease m1  | Response vector RMSE: 1.254366e-14
Increase m2  | Response vector RMSE: 2.019451e-14
Decrease m2  | Response vector RMSE: 1.636691e-14


**The least-squares decoder now reproduces all tested intervention responses to numerical precision.** Errors around \(10^{-14}\) are effectively zero here.

Our diagnosis is much clearer:

| Experiment | What we learned |
|---|---|
| Remove unit decoder normalization | The constraint contributed substantially to poor reconstruction. |
| Fit the decoder with least squares | The remaining error could be removed without changing the features. |
| Recheck interventions | Those same features support accurate intervention responses on this panel. |

**So this frozen encoder’s directions were sufficient for our toy task.** Its earlier poor performance wasn’t evidence that those directions couldn’t represent the required function.

**One qualification:** **W_enc**  stayed random and frozen, but the feature-producing biases had been trained. This result concerns that resulting representation, not completely untouched random features.

**This also explains the earlier slope errors:** the previous decoder combined the features incorrectly; a different decoder could correct those responses.



In [12]:
ls_checkpoint_path = checkpoint_dir / "free_decoder_least_squares.pt"

torch.save(
    {
        "state_dict": {
            name: tensor.detach().cpu().clone()
            for name, tensor in ls_state.items()
        },
        "config": diagnostic_checkpoint["models"]["Free decoder"]["config"],
        "source_checkpoint": diagnostic_path.name,
        "source_branch": "Free decoder",
        "source_total_epochs": 500,
        "method": "Float64 least squares with gelsd",
        "fitted_parameters": ["W_dec", "b_dec_out"],
        "fixed_parameters": ["W_enc", "b_enc", "b_dec"],
        "design_rank": int(fit.rank.item()),
        "design_columns": int(design.shape[1]),
        "data_seeds": {"train": 1001, "validation": 1002},
        "train_examples": len(x_vector_train),
        "validation_examples": len(x_vector_val),
        "objective": "Training MSE with fixed features; unrestricted decoder",
    },
    ls_checkpoint_path,
)

# Verify that the saved weights reload exactly.
saved_ls = torch.load(
    ls_checkpoint_path,
    map_location="cpu",
    weights_only=False,
)

for name, tensor in ls_state.items():
    assert torch.equal(saved_ls["state_dict"][name], tensor.cpu())

print("Saved and verified:", ls_checkpoint_path)

Saved and verified: /content/drive/MyDrive/transcoder_pilot/vector_run_seed2001/free_decoder_least_squares.pt


In [13]:
# Fresh held-out examples.
x_test, y_test = make_vector_data(2048, seed=1003)

# Keep all ±0.5 interventions within the original [-2, 2] range.
test_panel = x_test[(x_test.abs() <= 1.5).all(dim=-1)].clone()

test_results = {}

with torch.no_grad():
    for name, state in [
        ("Before least squares", free_state),
        ("After least squares", ls_state),
    ]:
        predictions, features = forward_from_state(state, x_test)

        mse = (predictions - y_test.to(predictions)).square().mean()
        l0 = (features > 0).float().sum(-1).mean()

        print(f"\n{name}")
        print(f"Test reconstruction MSE: {mse.item():.6e}")
        print(f"Test average L0: {l0.item():.2f}")
        print(f"Intervention examples: {len(test_panel)}")

        base_pred, _ = forward_from_state(state, test_panel)
        base_true = original_vector(test_panel)
        response_scores = {}

        for edit, (coordinate, amount) in edits.items():
            changed = test_panel.clone()
            changed[:, coordinate] += amount

            changed_pred, _ = forward_from_state(state, changed)
            true_response = original_vector(changed) - base_true
            predicted_response = changed_pred - base_pred

            error = predicted_response - true_response.to(predicted_response)
            rmse = error.square().sum(-1).mean().sqrt().item()

            response_scores[edit] = rmse
            print(f"{edit:12s} | Response vector RMSE: {rmse:.6e}")

        test_results[name] = {
            "reconstruction_mse": mse.item(),
            "average_l0": l0.item(),
            "response_rmse": response_scores,
        }

torch.save(
    {
        "test_seed": 1003,
        "test_examples": len(x_test),
        "intervention_examples": len(test_panel),
        "edits": edits,
        "results": test_results,
    },
    checkpoint_dir / "least_squares_test_results.pt",
)

print("\nTest results saved.")


Before least squares
Test reconstruction MSE: 3.473991e-02
Test average L0: 6.26
Intervention examples: 821
Identity     | Response vector RMSE: 0.000000e+00
Increase u   | Response vector RMSE: 1.642626e-01
Decrease u   | Response vector RMSE: 2.004113e-01
Increase m1  | Response vector RMSE: 8.670175e-02
Decrease m1  | Response vector RMSE: 5.921520e-02
Increase m2  | Response vector RMSE: 1.555373e-01
Decrease m2  | Response vector RMSE: 1.148487e-01

After least squares
Test reconstruction MSE: 3.526048e-27
Test average L0: 6.26
Intervention examples: 821
Identity     | Response vector RMSE: 0.000000e+00
Increase u   | Response vector RMSE: 1.415883e-14
Decrease u   | Response vector RMSE: 1.314768e-14
Increase m1  | Response vector RMSE: 1.284245e-14
Decrease m1  | Response vector RMSE: 1.211052e-14
Increase m2  | Response vector RMSE: 1.784086e-14
Decrease m2  | Response vector RMSE: 1.555773e-14

Test results saved.
